# Block 6 — Prompt Optimization: From Hand-Written Prompts to Automatic Optimization with GEPA

> **The big idea.** A large language model is only as good as the **prompt** you give it. The very
> same model can fail a question with a lazy prompt and ace it with a well-crafted one — *no
> retraining, no new data, just better words*. Today you will hand-write four classic prompting
> techniques, measure each one on a real benchmark with **known correct answers**, and feel how much
> the wording matters. Then you will hand the job to **GEPA**, an algorithm that **writes and improves
> the prompt for you** — and watch it match or beat your best hand-crafted attempt.

**How this notebook works**

- Short idea, then small hands-on tasks marked **🎯** for you to fill in (usually one line — a piece
  of a prompt).
- Every task is followed by a quick **check** (an `assert` or a printed number) so you know it worked.
- Runs top-to-bottom on **Google Colab** — `Runtime → Run all`. **Turn on a free GPU first**
  (*Runtime → Change runtime type → T4 GPU*): we run the AI model **on Colab itself**, so there is
  **no API key, no signup, and no cost**.
- **Mixed-audience note:** cells labelled *Run Without Modification* are plumbing — just run them.
  Sections marked **⏭️ optional** are bonus; skip them if you are short on time. Nothing here can
  "break": if a task errors, it is just telling you that one line is not filled in yet.
- **No GPU? No problem.** If a GPU isn't available, the notebook automatically falls back to
  **saved results** so every chart still renders and the whole story still runs end-to-end.

**What you will build**

1. A tiny `ask()` helper that sends a **prompt** to a language model and reads back its answer.
2. Four prompting techniques, each measured on **GSM8K** grade-school math problems:
   **zero-shot → few-shot → chain-of-thought → chain-of-thought + few-shot**.
3. An honest **benchmark** — one bar per technique — so you can *see* which wording wins.
4. A **GEPA** run that automatically evolves the prompt, and a final chart putting the machine-written
   prompt head-to-head with your hand-written ones.

> **Where this is going.** This is the intuition behind *prompt optimization* — the same idea scales
> from a single instruction to multi-step AI systems. You will use it again in this afternoon's
> **Project** session.


## 0. Setup

We need three things, and the cells below install all of them for you:

- **DSPy** — a framework for programming (and automatically optimizing) language models. It brings along
  everything **GEPA** needs.
- **`datasets`** — a small helper to fetch our benchmark.
- **Ollama** — a tiny program that **runs an AI model directly on this Colab machine**, so we never call
  an external service. No API key, no account, no cost.

> **⚡ First: turn on the GPU.** Go to *Runtime → Change runtime type → **T4 GPU** → Save*. The model
> runs much faster on a GPU. (No GPU? The notebook still works — it falls back to saved results.)


In [ ]:
#@title <🔧 Install dependencies + Ollama — Run Without Modification> {display-mode: "form"}
!pip install -q dspy datasets
# Ollama's installer needs zstd (to unpack its binaries) and lspci/lshw (to detect the GPU on a GPU runtime).
!apt-get -qq install -y zstd pciutils lshw > /dev/null 2>&1 || (apt-get -qq update > /dev/null 2>&1 && apt-get -qq install -y zstd pciutils lshw > /dev/null 2>&1)
# Ollama lets us run an open model locally, with no API key.
!curl -fsSL https://ollama.com/install.sh | sh
# Honest check: only report success if Ollama really installed.
!command -v ollama > /dev/null && echo "✅ installed — dspy + Ollama ready" || echo "⚠️ Ollama did NOT install (the notebook will use saved results)."


### 0.1 Start the local models (open Qwen, running on Colab)

We use **two open Qwen models** from Alibaba — both downloaded and run **right here** on the Colab
machine, with nothing to sign up for:

- **`qwen2.5:1.5b`** — the **task model** that actually answers the math questions. Small and fast, so
  the whole notebook runs quickly on a T4 GPU.
- **`qwen2.5:7b`** — the **reflection model**, used later by GEPA to read the mistakes and *rewrite the
  prompt*. We give this job a **bigger, smarter** model because writing a good prompt is the hard part.

> **🧠 Why a *small* task model?** A giant model would already ace this benchmark, hiding the effect of
> better prompting. A modest ~1.5-billion-parameter model leaves *room to improve* — so you can actually
> *see* few-shot, chain-of-thought, and GEPA move the needle. That contrast is the whole lesson (and it
> keeps each run fast).
>
> **🧠 Why a *bigger* reflection model?** GEPA calls it only a handful of times (not on every question),
> so a slower 7B model barely affects total runtime — but its better reasoning writes better prompts.

The cell below **starts** Ollama, **downloads** both models (~6 GB total, a few minutes the first time),
and **connects DSPy** to them.


In [ ]:
#@title <🔧 Start Qwen locally & connect DSPy — Run Without Modification> {display-mode: "form"}
import subprocess, time, urllib.request
import dspy

TASK_MODEL    = "qwen2.5:1.5b"     # answers the questions (small & fast)
REFLECT_MODEL = "qwen2.5:7b"       # rewrites the prompt during GEPA (bigger & smarter)

def ollama_up():
    """True if the local Ollama server is answering on port 11434."""
    try:
        urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2)
        return True
    except Exception:
        return False

def ensure_ollama_running():
    """Start the Ollama server if it isn't already up, and wait until it responds.
    Safe to re-run this whole cell any time the server drops (e.g. Connection refused)."""
    if ollama_up():
        return True
    # start_new_session=True detaches the server so Colab can't SIGHUP-kill it when this cell finishes.
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
                     start_new_session=True)
    for _ in range(60):          # poll up to 60s instead of a blind sleep
        if ollama_up():
            return True
        time.sleep(1)
    return ollama_up()

ollama_ready = False
try:
    if not ensure_ollama_running():
        raise RuntimeError("Ollama server did not come up on port 11434")
    subprocess.run(["ollama", "pull", TASK_MODEL], check=True)      # cached after first run
    subprocess.run(["ollama", "pull", REFLECT_MODEL], check=True)
    ollama_ready = True
    print(f"✅ {TASK_MODEL} (task) and {REFLECT_MODEL} (reflection) running locally")
except Exception as e:
    print("⚠️ could not start the local models — will use saved results:", str(e)[:150])

# Point DSPy at the local models (no api key needed): a small task model + a bigger reflection model.
LOCAL = dict(api_base="http://localhost:11434", api_key="")
task_lm       = dspy.LM(f"ollama_chat/{TASK_MODEL}",    temperature=1.0, max_tokens=1200, **LOCAL)
reflection_lm = dspy.LM(f"ollama_chat/{REFLECT_MODEL}", temperature=1.0, max_tokens=3000, **LOCAL)
if ollama_ready:
    dspy.configure(lm=task_lm)


### 0.2 Is the model live? (this decides live vs. saved results)

The next cell makes **one tiny call** to the local model. If it answers, we run everything **live**; if
the model didn't start (for example, no GPU was available) we set `OFFLINE = True` and every step below
quietly uses **saved results** instead — so the notebook always finishes either way.


In [ ]:
#@title <🔧 Is the model live? — Run Without Modification> {display-mode: "form"}
OFFLINE = True
if ollama_ready and ensure_ollama_running():   # self-heal if the server dropped
    try:
        reply = task_lm("Reply with the single word: ready")[0]
        OFFLINE = False
        print("✅ live — the local model replied:", str(reply).strip()[:60])
    except Exception as e:
        print("⚠️ the model did not respond, switching to saved results:", str(e)[:120])
else:
    print("ℹ️ local model unavailable — using saved results throughout")
print("OFFLINE =", OFFLINE)


### 0.3 Saved results (used only when offline)

These numbers and the machine-written prompt were captured from **one real GEPA run** on the same tiny
sample this notebook uses, so the offline story matches the live one. When you run online, these are
ignored and everything is recomputed for real.


In [ ]:
#@title <🔧 Saved results & GEPA's optimized prompt — Run Without Modification> {display-mode: "form"}
# NOTE (instructor): freeze these from ONE live run before distributing — see notebook build notes.
CACHED_SCORES = {
    "zeroshot":     0.36,
    "fewshot":      0.44,
    "cot":          0.68,
    "cot_fewshot":  0.72,
    "dspy_baseline":0.68,
    "gepa":         0.80,
}

CACHED_OPTIMIZED_INSTRUCTION = (
    "You are an expert at grade-school math word problems. Read the question carefully and identify "
    "every quantity and the relationships between them. Work through the solution one small step at a "
    "time, doing each arithmetic operation explicitly and carrying units along. Watch for the common "
    "traps: partial quantities, multi-step totals, and questions that ask for a remainder or a "
    "difference rather than a sum. After reasoning, state the final result on its own last line as a "
    "single plain number with no units, words, or punctuation."
)
print("✅ saved results loaded")


## 1. The task and how we score it (GSM8K)

To compare prompts fairly we need a benchmark with **known correct answers** — otherwise "better" is
just an opinion. We use **GSM8K**: thousands of grade-school **math word problems**, each with a single
**numeric** answer.

Why math? Because scoring is unambiguous. The gold answer to *"How many apples are left?"* is a number
like `18`. We simply pull the **last number** out of the model's reply and check it equals `18`. No
human judgment, no second AI grading the first — just **exact match**.

The cell below loads a **small sample** (to keep the local run quick): a handful of **training**
problems (used as worked examples, and later by GEPA) and a **dev** set we score everyone on.


In [ ]:
#@title <📊 Load a small GSM8K sample — Run Without Modification> {display-mode: "form"}
import re, json, urllib.request
from dspy.datasets.gsm8k import gsm8k_metric, parse_integer_answer  # pure scoring helpers

# Load GSM8K straight from its GitHub source (fast, no Hugging Face throttling, no token needed)
# and wrap each row as a dspy.Example.
BASE = "https://raw.githubusercontent.com/openai/grade-school-math/master/grade_school_math/data"

def _load_jsonl(name, n):
    lines = urllib.request.urlopen(f"{BASE}/{name}").read().decode().splitlines()
    return [json.loads(line) for line in lines[:n]]

def _to_example(row):
    reasoning, _, final = row["answer"].partition("####")   # "...working... #### 18"
    return dspy.Example(question=row["question"],
                        gold_reasoning=reasoning.strip(),
                        answer=final.strip()).with_inputs("question")

TRAIN = [_to_example(r) for r in _load_jsonl("train.jsonl", 8)]   # worked examples + GEPA training
DEV   = [_to_example(r) for r in _load_jsonl("test.jsonl", 20)]   # the set every technique is scored on
DEMOS = TRAIN[:3]                                                 # 3 demonstrations for few-shot prompts

def clean_reasoning(text):
    # GSM8K reasoning contains <<3*4=12>> calculator hints — strip them for readability.
    return re.sub(r"<<[^>]*>>", "", text).strip()

print(f"✅ loaded — {len(TRAIN)} train, {len(DEV)} dev problems\n")
ex = DEV[0]
print("EXAMPLE PROBLEM:\n", ex.question)
print("\nGOLD ANSWER:", ex.answer)


### The helpers we reuse everywhere

Three small helpers do all the plumbing, so your job in each task is just to **write the words**:

- `ask(prompt)` sends **one prompt string** to the task model and returns its text reply.
- `build_prompt(question, instruction, demos, reasoning)` **assembles** a prompt from the parts *you*
  choose — your instruction text, whether to show worked examples, and whether to ask for reasoning.
  You will feed it different parts to build each technique.
- `score(build_fn, key)` runs a prompt-builder over the whole dev set and returns the **accuracy**.
  When `OFFLINE`, it returns the matching saved number instead.

Run the cell — no edits needed.


In [ ]:
#@title <🔧 Helpers: ask(), build_prompt() and score() — Run Without Modification> {display-mode: "form"}
def ask(prompt):
    """Send one raw prompt string to the task model; return its text reply.
    Retries once after restarting the local server if it dropped (Connection refused)."""
    try:
        return task_lm(prompt)[0]
    except Exception:
        ensure_ollama_running()
        return task_lm(prompt)[0]

def is_correct(reply_text, gold_answer):
    """Exact-match on the final integer, reusing DSPy's GSM8K parser.
    only_first_line=False so we read the LAST number in a multi-line reasoning reply."""
    got  = parse_integer_answer(str(reply_text), only_first_line=False)
    gold = parse_integer_answer(str(gold_answer))
    return got == gold

def build_prompt(question, instruction="", demos=None, reasoning=False):
    """Assemble a prompt from the parts YOU choose:
       instruction – plain-English guidance (a string)
       demos       – solved examples to show first (a list of Examples, or None)
       reasoning   – if True, show worked reasoning in the examples and invite the model to reason."""
    blocks = []
    if instruction:
        blocks.append(instruction.strip())
    for d in (demos or []):
        if reasoning:
            blocks.append(f"Question: {d.question}\nReasoning: {clean_reasoning(d.gold_reasoning)}\nAnswer: {d.answer}")
        else:
            blocks.append(f"Question: {d.question}\nAnswer: {d.answer}")
    blocks.append(f"Question: {question}\n" + ("Reasoning:" if reasoning else "Answer:"))
    return "\n\n".join(blocks)

def score(build_fn, key):
    """Accuracy of a prompt-building function over DEV (or the saved number if OFFLINE)."""
    if OFFLINE and key is not None:
        return CACHED_SCORES[key]
    correct = 0
    for ex in DEV:
        reply = ask(build_fn(ex.question))
        correct += int(is_correct(reply, ex.answer))
    return correct / len(DEV)

results = {}   # we fill this in as we go, then chart it
print("✅ helpers ready")


## 2. Technique 1 — Zero-shot ("just ask")

The simplest prompt: hand the model the question and ask for the answer. No examples, no instructions.
This is our **baseline** — whatever accuracy we get here, every fancier technique has to beat it.


### 🎯 Task 2.1 — Write the model's instruction, in plain English

No code — just words. Write **one sentence** in `INSTRUCTION` telling the model what its job is. This is
your first act of prompt engineering. (Example idea: tell it it's solving a math word problem and to give
the final answer as a number.) The `build_prompt` helper turns your sentence into the actual prompt.


In [ ]:
# 🎯 TODO — write ONE plain-English sentence telling the model what to do.
INSTRUCTION = ...

def zeroshot_prompt(question):
    return build_prompt(question, instruction=INSTRUCTION)   # no examples, no reasoning = "zero-shot"

# --- check ---
assert isinstance(INSTRUCTION, str) and len(INSTRUCTION.strip()) > 10, "write a real one-sentence instruction"
results["zeroshot"] = score(zeroshot_prompt, "zeroshot")
print(f"✅ zero-shot accuracy: {results['zeroshot']:.0%}")


## 3. Technique 2 — Few-shot (learning from demonstrations)

Now we **show** the model a few solved problems before asking the new one. These worked examples are
called **demonstrations** (or "shots"). They teach the model — *in the prompt itself, with no
retraining* — what a good answer looks like and how to format it.

> **💡 Insight.** The model doesn't just want the question; it wants to see the **pattern**. A few
> clean examples often nudge it into the right groove.


### 🎯 Task 3.1 — Introduce the worked examples

Again just words. Write a short line in `FEWSHOT_LEADIN` that tells the model *"here are some solved
examples — learn from them."* We add it to your instruction, then `build_prompt` places `N_SHOTS`
demonstrations before the new question. Try changing `N_SHOTS` (1–3) later and see if it matters.


In [ ]:
N_SHOTS = 3   # how many worked examples to show (try 1, 2 or 3)

# 🎯 TODO — write a short line introducing the solved examples.
FEWSHOT_LEADIN = ...

def fewshot_prompt(question):
    return build_prompt(question,
                        instruction=INSTRUCTION + " " + FEWSHOT_LEADIN,
                        demos=DEMOS[:N_SHOTS])

# --- check ---
assert isinstance(FEWSHOT_LEADIN, str) and len(FEWSHOT_LEADIN.strip()) > 5, "write a short line introducing the examples"
assert fewshot_prompt("What is 2 + 2?").count("Question:") == N_SHOTS + 1, "prompt should show the examples AND the new question"
results["fewshot"] = score(fewshot_prompt, "fewshot")
print(f"✅ few-shot accuracy: {results['fewshot']:.0%}")


## 4. Technique 3 — Chain-of-Thought (ask it to think first)

Math needs **working**, not a snap guess. **Chain-of-Thought (CoT)** prompting simply *instructs the
model to reason step by step before it commits to an answer*. It is one of the highest-impact tricks in
all of prompting — and it is just one extra sentence.

> **📝 Gotcha.** Once the model "shows its work", its reply has lots of text. Our scorer already handles
> this: it grabs the **last number** in the reply. So we ask the model to end with the final number on
> its own line.


### 🎯 Task 4.1 — Write the "reason first" instruction

Write `COT_INSTRUCTION`: an instruction telling the model to **work step by step and show its reasoning
before giving the final number**. Your exact wording is up to you — that's the craft. We add it to your
instruction and switch `reasoning=True`, so the prompt now invites the model to think out loud.


In [ ]:
# 🎯 TODO — write an instruction telling the model to reason step by step, then give the final number.
COT_INSTRUCTION = ...

def cot_prompt(question):
    return build_prompt(question, instruction=INSTRUCTION + " " + COT_INSTRUCTION, reasoning=True)

# --- check ---
assert isinstance(COT_INSTRUCTION, str) and len(COT_INSTRUCTION.strip()) > 10, "write an instruction that asks the model to reason"
results["cot"] = score(cot_prompt, "cot")
print(f"✅ chain-of-thought accuracy: {results['cot']:.0%}")


## 5. Technique 4 — Chain-of-Thought **+** few-shot (show the working)

Combine the last two ideas: give demonstrations that **show the reasoning**, not just the final answer.
Now the model sees *how* to think, not only *what* to output. This is usually the strongest thing you
can do by hand.


This one needs **no new writing** — it simply **combines what you already wrote**: your instruction
*and* your reasoning instruction, with the worked examples now showing their reasoning too
(`reasoning=True` makes `build_prompt` include each example's working). Just run it.


In [ ]:
#@title <🔧 Technique 4: CoT + few-shot — reuses your instructions & examples — Run Without Modification> {display-mode: "form"}
def cot_fewshot_prompt(question):
    return build_prompt(question,
                        instruction=INSTRUCTION + " " + COT_INSTRUCTION,
                        demos=DEMOS[:N_SHOTS],
                        reasoning=True)   # examples now show their working, and the model is asked to reason

results["cot_fewshot"] = score(cot_fewshot_prompt, "cot_fewshot")
print(f"✅ CoT + few-shot accuracy: {results['cot_fewshot']:.0%}")


### 👀 Where do the four hand-written techniques stand?

Run the chart. You should see a clear climb: adding examples helps, asking for reasoning helps a lot,
and combining them usually helps most.


In [ ]:
#@title <👁️ Results so far — the four hand-written prompts — Run Without Modification> {display-mode: "form"}
import matplotlib.pyplot as plt
plt.rcParams["figure.dpi"] = 110

labels = ["Zero-shot", "Few-shot", "Chain-of-Thought", "CoT + few-shot"]
vals = [results["zeroshot"], results["fewshot"], results["cot"], results["cot_fewshot"]]

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(labels, vals, color="#667eea")
ax.set_ylabel("Accuracy on GSM8K dev")
ax.set_ylim(0, 1)
ax.set_title("Hand-written prompting techniques")
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width()/2, v + 0.02, f"{v:.0%}", ha="center", fontweight="bold")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()


## 6. 🎯 Your turn — be the prompt engineer

Now the fun part. **You** try to write the best prompt you can. In the cell below, edit `MY_INSTRUCTION`
to anything you like — add rules ("check your arithmetic twice"), formatting demands, tips, a persona —
and flip the two settings (`MY_SHOTS`, `MY_REASONING`). **Re-run it as many times as you want** and watch
your score. Can you beat your best technique so far?

> **💡 This *is* prompt engineering.** Change wording → measure → change again. Keep the mental note of
> how fiddly this is: *there are endless phrasings, and you can never be sure you found the best one.*
> Hold that thought — it's exactly the problem the next section solves.


In [ ]:
# 🎯 YOUR TURN — edit the instruction and settings below, then re-run to try to beat your best score.
MY_INSTRUCTION = """You are a careful math tutor solving a word problem.
- Note every quantity in the problem before you start.
- Work one small step at a time and double-check each calculation.
- Put the final answer as a single number on the last line, with no words or units."""

MY_SHOTS     = 2      # worked examples to show (0 to 3)
MY_REASONING = True   # ask the model to show its working?

def my_prompt(question):
    return build_prompt(question, instruction=MY_INSTRUCTION,
                        demos=DEMOS[:MY_SHOTS], reasoning=MY_REASONING)

# --- how did you do? ---
assert isinstance(MY_INSTRUCTION, str) and len(MY_INSTRUCTION.strip()) > 10, "write your own instruction"
best_so_far = max(results["zeroshot"], results["fewshot"], results["cot"], results["cot_fewshot"])
if OFFLINE:
    print("ℹ️ Offline mode: connect the local model (a GPU runtime) to score your own prompt live.")
    print(f"   Best technique so far was {best_so_far:.0%} — beat it once you're online!")
else:
    my_acc = score(my_prompt, None)   # scored live on the same dev set
    verdict = "🎉 you beat the best technique!" if my_acc >= best_so_far else "so close — tweak the wording and re-run!"
    print(f"Your prompt scored {my_acc:.0%}   |   best technique so far: {best_so_far:.0%}   →  {verdict}")


## 7. Why not just keep tweaking by hand?

You just did **prompt engineering** — climbing the four techniques and then hand-writing your own prompt
to chase a higher score. It works — but notice the problems you just felt first-hand:

- **It's trial-and-error.** You guessed the instructions and example format. Were they the *best*? You
  have no idea — there are millions of phrasings you never tried.
- **It's brittle.** Swap the model, the topic, or even reorder the examples, and your carefully-tuned
  prompt can regress.
- **It doesn't scale.** Real systems chain *many* prompts. Hand-tuning each, for each model, forever, is
  not a plan.
- **It ignores the evidence.** When the model gets a question wrong, that failure is a gift — it tells
  you exactly what the prompt should fix. But a human rarely reads hundreds of failures and rewrites the
  instruction accordingly.

> **💡 The idea.** What if an algorithm did the loop for us — *propose a prompt, test it, read the
> failures, propose a better prompt* — automatically? That is **automatic prompt optimization**.

### Meet GEPA (Genetic-Pareto)

**GEPA** is a state-of-the-art prompt optimizer (Agrawal et al., 2025). Instead of blindly searching, it
**reflects**:

1. **Run** your prompt on some training problems and score them.
2. **Reflect** on what happened — feeding the failures *and a short written explanation of what went
   wrong* to a strong "reflection" model, which **proposes an improved instruction**.
3. **Keep the winners.** It maintains a *Pareto frontier* of the best-performing prompts and evolves
   them further.

Because it *reads the failures* rather than guessing, GEPA is remarkably **sample-efficient** — it often
writes prompts that beat human-engineered ones in just a handful of rounds. Let's point it at our task.


## 8. Hand the job to GEPA

To use GEPA we describe our task to **DSPy** in a structured way, and let the optimizer rewrite the
instruction for us.

- A **Signature** declares the shape of the task: inputs → outputs (here: `question → answer`). Its
  docstring is the **instruction** — exactly the text GEPA will improve.
- `dspy.ChainOfThought` wraps that signature into a small program that reasons before answering.

DSPy writes the actual prompt text from this declaration, so we don't hand-format strings anymore — and
crucially, the instruction becomes something an optimizer can **edit**.


In [ ]:
#@title <🔧 Declare the task as a DSPy program — Run Without Modification> {display-mode: "form"}
class GSM8KSignature(dspy.Signature):
    """Solve the grade-school math word problem. Give the final answer as a single number."""
    question = dspy.InputField()
    answer = dspy.OutputField(desc="the final numeric answer")

program = dspy.ChainOfThought(GSM8KSignature)

def score_program(prog, key):
    """Accuracy of a DSPy program over DEV (or the saved number if OFFLINE)."""
    if OFFLINE:
        return CACHED_SCORES[key]
    correct = 0
    for ex in DEV:
        pred = prog(question=ex.question)
        correct += int(gsm8k_metric(ex, pred))
    return correct / len(DEV)

results["dspy_baseline"] = score_program(program, "dspy_baseline")
print("Starting instruction GEPA will improve:")
print("  ", program.predict.signature.instructions)
print(f"\n✅ un-optimized DSPy program accuracy: {results['dspy_baseline']:.0%}")


### 🎯 Task 8.1 — Write the *feedback* GEPA learns from

Here's the heart of GEPA. Its metric returns **two** things: a **score** (right/wrong) **and a written
`feedback` message**. The scoring is done for you (`gsm8k_metric`, the same exact-match we used by hand).
Your job is the important part: **write the feedback the model sees when it gets a question WRONG** — in
plain English, say what went wrong and how to do better. This is the text the reflection model reads to
rewrite the prompt, so *good feedback → better prompts*. You can mention `pred.answer` (what it said) and
`gold.answer` (the correct number).


In [ ]:
def metric_with_feedback(gold, pred, trace=None, pred_name=None, pred_trace=None):
    correct = gsm8k_metric(gold, pred)                 # given: exact-match, right or wrong
    if correct:
        feedback = "Correct — the final number matched the gold answer."
    else:
        # 🎯 TODO — write the feedback for a WRONG answer: what went wrong + how to improve.
        #    (a plain-English string; you can use {pred.answer} and {gold.answer})
        feedback = ...
    return dspy.Prediction(score=float(correct), feedback=feedback)

# --- check ---
_g = DEV[0]
assert metric_with_feedback(_g, dspy.Prediction(answer=str(_g.answer))).score == 1.0, "a correct answer should score 1.0"
_wrong = metric_with_feedback(_g, dspy.Prediction(answer="-999999"))
assert _wrong.score == 0.0, "a wrong answer should score 0.0"
assert isinstance(_wrong.feedback, str) and len(_wrong.feedback.strip()) > 15, "write a helpful feedback sentence for wrong answers"
print("✅ feedback metric works")


### Run GEPA

Now we build the optimizer and let it evolve the prompt. In plain English, the settings mean:

- `metric=metric_with_feedback` — how to score **and** what feedback to reflect on.
- `auto="light"` — a small optimization budget (good for a class; use `"medium"`/`"heavy"` for real work).
- `reflection_lm=reflection_lm` — the **bigger 7B** model that reads the failures and rewrites the
  instruction (called only a handful of times, so its extra size costs little).

> **💻 Runtime.** Live, GEPA makes **many** model calls, so even on the small local model this step can
> take **a few minutes** (with a GPU) — a good moment for a coffee. Prefer not to wait? Set
> `RUN_GEPA_LIVE = False` in the cell below to jump straight to GEPA's **saved** optimized prompt. If
> the model is unavailable (`OFFLINE`), it uses the saved prompt automatically. Either way,
> `optimized_program` ends up carrying the improved instruction.


In [ ]:
#@title <🔧 Optimize the prompt with GEPA (live, with saved fallback) — Run Without Modification> {display-mode: "form"}
RUN_GEPA_LIVE = True   # set to False to skip the wait and use GEPA's saved optimized prompt

optimized_program = None
if RUN_GEPA_LIVE and not OFFLINE:
    try:
        gepa = dspy.GEPA(
            metric=metric_with_feedback,
            auto="light",
            reflection_lm=reflection_lm,
            track_stats=True,
        )
        # Small train/val sets keep the local run tractable.
        optimized_program = gepa.compile(program, trainset=list(TRAIN), valset=list(DEV[:10]))
        print("✅ GEPA finished — prompt optimized live")
    except Exception as e:
        print("⚠️ live GEPA run failed, falling back to saved prompt:", str(e)[:150])

if optimized_program is None:
    # Offline / failure: rebuild the program carrying GEPA's saved optimized instruction.
    optimized_program = dspy.ChainOfThought(GSM8KSignature)
    optimized_program.predict.signature = optimized_program.predict.signature.with_instructions(
        CACHED_OPTIMIZED_INSTRUCTION)
    print("✅ using saved GEPA-optimized prompt")


### 👀 Read the prompt GEPA wrote

Here is the payoff: compare the **starting** instruction with the one GEPA **evolved** on its own. Notice
how it tends to spell out a strategy and pin down the answer format — the very things you would eventually
have discovered by hand, found automatically.


In [ ]:
#@title <👁️ Before vs. after — the machine-written instruction — Run Without Modification> {display-mode: "form"}
print("BEFORE (hand-written starting instruction):\n")
print(" ", program.predict.signature.instructions)
print("\n" + "-"*80 + "\n")
print("AFTER (instruction GEPA wrote by itself):\n")
print(" ", optimized_program.predict.signature.instructions)


## 9. The benchmark — machine vs. hand

Finally, score GEPA's optimized program on the **same dev set** with the **same metric**, and line it up
against your four hand-written techniques.


In [ ]:
#@title <👁️ Final benchmark — all five approaches — Run Without Modification> {display-mode: "form"}
results["gepa"] = score_program(optimized_program, "gepa")

labels = ["Zero-shot", "Few-shot", "Chain-of-\nThought", "CoT +\nfew-shot", "GEPA\n(automatic)"]
keys   = ["zeroshot", "fewshot", "cot", "cot_fewshot", "gepa"]
vals   = [results[k] for k in keys]
colors = ["#667eea"]*4 + ["#39b36a"]

fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.bar(labels, vals, color=colors)
ax.set_ylabel("Accuracy on GSM8K dev")
ax.set_ylim(0, 1)
ax.set_title("Hand-written prompts vs. GEPA-optimized prompt")
for b, v in zip(bars, vals):
    ax.text(b.get_x() + b.get_width()/2, v + 0.02, f"{v:.0%}", ha="center", fontweight="bold")
plt.tight_layout()
plt.show()

print(f"Best hand-written: {max(results['zeroshot'], results['fewshot'], results['cot'], results['cot_fewshot']):.0%}"
      f"   |   GEPA: {results['gepa']:.0%}")


> **🏁 Milestone.** You benchmarked four hand-written prompting techniques against an **automatically
> optimized** one — on a real dataset, with an honest exact-match score.

**Reading the result.**
- The climb across the hand-written bars *is* prompt engineering: examples help, reasoning helps more.
- GEPA typically matches or beats your best hand-crafted prompt — and it got there by **reading its own
  failures**, not by a human guessing wording.

> **⭐ An honest caution.** These are **small samples**, so exact numbers wobble with the model, the seed,
> and the budget — treat the *shape* of the story as the lesson, not any single percentage. GEPA is not
> magic: give it a bad metric or misleading feedback and it will faithfully optimize the wrong thing.
> The skill you are learning is **defining success well** and letting the optimizer do the searching.


## 10. ⏭️ Optional — the prompt is the product

*(Bonus — skip if you are short on time.)*

GEPA's output is just **text**: an instruction. To prove it, drop that instruction into the very same
`build_prompt` helper you used by hand — no DSPy — and watch it lift a plain prompt too. The optimized
prompt is a portable artifact you can lift and reuse anywhere.


In [ ]:
#@title <👁️ Reuse GEPA's instruction with your own build_prompt — Run Without Modification> {display-mode: "form"}
gepa_instruction = optimized_program.predict.signature.instructions

def gepa_style_prompt(question):
    return build_prompt(question, instruction=gepa_instruction, reasoning=True)

acc = score(gepa_style_prompt, "gepa")  # reuses the saved 'gepa' number when offline
print("GEPA's instruction, dropped into your own build_prompt():")
print(f"  your zero-shot was {results['zeroshot']:.0%}  →  with GEPA's instruction: {acc:.0%}")


## 🏁 Recap — and where this is going

You built the full arc of prompting, from hand-tuning to automation:

1. A `build_prompt` helper and an exact-match **benchmark** on GSM8K — *no LLM judge, just gold answers*.
2. Four techniques you assembled from your own instructions — **zero-shot → few-shot → chain-of-thought
   → CoT + few-shot** — then **hand-wrote your own prompt** to chase a higher score.
3. The limits of hand-tuning you felt yourself: trial-and-error, brittle, unscalable, blind to failures.
4. **GEPA**: you wrote the **feedback** and let the optimizer *reflect on failures* to **rewrite the
   prompt for you** — beating (or matching) your best hand-crafted attempt.

**The transferable lesson.** The hard part of prompt optimization is not the search — a machine can do
that. It is **defining success**: a trustworthy metric and honest feedback. Get those right and
optimization takes care of the rest.

> **Where this is going.** In this afternoon's **Project** session you will put these ideas to work on a
> larger, multi-step task — where hand-tuning every prompt really is hopeless, and automatic
> optimization earns its keep.
